# 06 Gitee远端同步、分支与代码审查


## 1. 本篇对应的课程与资料范围

上一篇已经建立本地 Git 存档，并区分了代码回退、会话回退和持久化记忆。本篇继续解决两个问题：怎样把已提交的代码保存到 Gitee，以及怎样在独立分支上修改、审查和交付代码。

按课程视频的顺序，对应以下两节：

| 视频 | 本篇主题 |
|---|---|
| `14-工程化控制-将代码推送到Gitee远端服务器.mp4` | 将本地提交同步到远端仓库 |
| `15-工程化控制-创建代码分支和执行代码审查.mp4` | 分支开发与代码审查 |

课程文件见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`；配套讲义是 [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p)。本篇主要使用讲义第 3 节的分支示例、第 5 节“代码审查”和附录的典型工作流。

**资料边界：**已核对视频名称，并读取相关讲义正文和原始代码块；视频预览跳回分享目录，未取得可阅读的逐段内容。讲义没有展开 Gitee 建仓、认证和首次推送的课堂细节，因此下面这些操作根据 Git、Gitee 官方文档整理，明确标为“补充”，不视为视频转录。实际课堂仓库地址、分支名和审查结果均未确认。

讲义标注适用版本为“截至 2026 年 7 月”。本篇对审查命令的范围、别名和发布能力作就地核对，核对日期为 `2026-10-10`。后续 Skills、Agent、Hook 的开发留在对应笔记。


## 2. 补充：commit、push、分支分别解决什么

本地 `commit` 把暂存区内容记录为一个版本；`push` 把指定分支上的提交及所需对象传到远端。只修改文件、只执行 `git add`，都还没有产生可推送的新版本。

| 名称 | 含义 | 在本篇中的作用 |
|---|---|---|
| 工作区 | 正在编辑的文件 | 承载本次修改 |
| 暂存区 | 下一次提交准备包含的内容 | 确定提交范围 |
| 本地分支 | 指向某条提交历史末端的引用 | 将功能开发与主线分开推进 |
| `origin` | 一个远端仓库的简称 | 对应自己的 Gitee 仓库地址 |
| `origin/main` | 本地记录的远端 `main` 分支位置 | 通过 fetch 等操作更新后用于比较 |
| upstream，上游分支 | 本地分支关联的跟踪目标 | 帮助 Git 判断相对哪个分支拉取、推送和比较 |

`origin` 不是账号名，也不是 Gitee 专用关键字；`main` 也不是固定要求，有些仓库使用 `master`。下面命令以 `main` 为主分支示例，执行前应将它统一替换为实际主分支名。

远端保存的是 Git 已跟踪并提交的内容。记账应用运行时写入 IndexedDB 的个人账目，不会因为推送源码就自动上传或跨设备同步；远端存档也不会自动部署桌面应用。


## 3. 补充：推送前确认项目根目录和本地版本

下面所有 Git 操作都在**黑马记账应用的项目根目录**执行，也就是含 `package.json`、`src/`、`src-tauri/` 的目录。不要在存放这篇学习笔记的仓库里照抄执行。

先查看当前状态：

```bash
git rev-parse --show-toplevel
git status
git branch --show-current
git log --oneline -5
git remote -v
```

依次确认：Git 管理的是记账项目；当前分支名正确；已经存在本地提交；是否已有远端。

如果 `git log` 提示尚无提交，应先完成 05 中的初始化、选择文件和提交步骤。如果还有应保存的修改，先查看 `git diff`，处理后再提交；不要把“已添加远端”理解成“文件已备份”。

沿用前篇的忽略规则，通常不提交 `node_modules/`、`dist/`、`src-tauri/target/` 和打包生成的可执行文件。源代码、配置、锁文件和需要的资源应随项目保存。

`.gitignore` 对已经被 Git 跟踪的文件不会自动生效。可以用 `git ls-files` 检查实际跟踪范围；推送前还应确认提交历史里没有误收录的本地凭据和无关个人文件。


## 4. 补充：在 Gitee 创建远端仓库并确定认证方式

首次同步的前提是：本地已有提交，远端是自己有写权限的仓库。

1. 登录 Gitee，新建用于保存该项目的仓库。
2. 确认仓库所属账号或组织、仓库路径和可见性。
3. 若要直接推送现有本地历史，创建**空仓库**；不要同时在远端初始化 README、许可证或 `.gitignore`，以免产生另一条初始历史。
4. 从仓库页面复制克隆地址。仓库显示名称和 URL 路径可以不同，应复制页面提供的实际地址。

两种常见地址形式如下，仅用于辨认格式，`YOUR_NAMESPACE`、`YOUR_REPO` 都要替换：

```text
HTTPS: https://gitee.com/YOUR_NAMESPACE/YOUR_REPO.git
SSH:   git@gitee.com:YOUR_NAMESPACE/YOUR_REPO.git
```

HTTPS 按 Gitee 当前认证提示和本机凭据管理器完成验证；不要将密码或令牌直接拼进仓库 URL。网页已经登录，不等于终端已经拥有 Git 推送凭据。

使用 SSH 时，需要将公钥绑定到有仓库写权限的账号。Gitee 官方区分**账户 SSH 公钥**和**仓库部署公钥**：账户公钥配合账号权限支持推送和拉取，部署公钥通常用于只读拉取，不能直接当成开发者写权限。

需要配置时参见 [Gitee：账户 SSH 公钥设置](https://help.gitee.com/base/account/SSH%E5%85%AC%E9%92%A5%E8%AE%BE%E7%BD%AE)。已有密钥时先检查，避免覆盖。只添加 `.pub` 公钥内容，私钥保留在本机。

配置好后可检查 SSH 认证：

```bash
ssh -T git@gitee.com
```

官方说明中，成功响应会包含绑定的用户名，并说明不提供 shell access；这条提示不代表失败。首次连接应按官方说明核对主机身份。认证成功后，仍需对目标仓库具有写权限。


## 5. 补充：把本地仓库连接到 origin

在项目根目录先执行 `git remote -v`，根据结果选择操作。

**还没有 `origin`：**将下方 URL 替换为自己复制的 HTTPS 地址后添加：

```bash
git remote add origin "https://gitee.com/YOUR_NAMESPACE/YOUR_REPO.git"
git remote -v
```

如果采用 SSH，就在上面的同一位置使用自己的 SSH 地址；两种方式选一种即可。

**已经有 `origin`：**如果地址正确，直接复用；只有明确要更换当前远端时，才修改它：

```bash
git remote set-url origin "https://gitee.com/YOUR_NAMESPACE/YOUR_REPO.git"
git remote -v
```

`remote add` 只是写入远端配置，不会上传文件。`remote origin already exists` 表示该名称已有配置，应先检查 URL，不能靠重复添加解决。

检查结果中的 fetch 和 push 地址是否都指向预期仓库。如果原项目还需要保留另一个远端，应使用不同简称并明确推送目标，不能直接覆盖现有配置。


## 6. 补充：首次推送与远端结果核对

以下示例前提为：当前要同步的本地主分支叫 `main`，已存在至少一次提交，`origin` 指向空的 Gitee 仓库。实际叫 `master` 时，将本节所有 `main` 替换为 `master`。

```bash
git push -u origin main
```

这条命令把本地 `main` 推到远端同名分支；`-u` 同时建立上游跟踪关系。后续同一分支在默认推送配置下可直接 `git push`，但学习时写明远端和分支更容易核对目标。

推送结束后，在本地查看：

```bash
git status -sb
git branch -vv
git log -1 --oneline
```

再打开 Gitee 仓库，切换到对应分支，确认能看到源代码、提交说明和刚才的提交。仓库里出现文件但停在旧提交，不能算本次同步成功。

需要精确比较时：

```bash
git rev-parse main
git ls-remote origin refs/heads/main
```

第一条显示本地主分支的完整提交 ID；第二条直接查询服务器上的该分支引用。两者的提交 ID 应一致。

`git status` 中的“工作区干净”只说明本地文件与提交的关系；“与上游一致”依赖本地已知的远端状态，不能独自证明服务器上当前是什么版本。


## 7. 补充：区分推送失败、拉取和历史分叉

遇到错误时，先根据错误内容定位，避免把所有问题都当成网络问题。

| 提示或现象 | 重点检查 |
|---|---|
| `src refspec main does not match any` | 本地是否真的有 `main`，是否已经存在提交 |
| `remote origin already exists` | 远端简称已存在，检查地址后复用或有目的地修改 |
| Authentication failed / Permission denied | HTTPS 凭据或 SSH 账号认证，以及仓库写权限 |
| `non-fast-forward` / rejected | 远端有本地尚未整合的提交，先读取和比较历史 |
| 仓库体积或文件大小被拒绝 | 是否误提交依赖、构建产物或大文件；删除工作区文件不等于清除历史 |

怀疑远端已经更新时，先抓取，再比较：

```bash
git fetch origin
git log --oneline --graph --decorate --all -15
git status -sb
```

`fetch` 更新本地保存的远端引用，不会把远端改动自动合入当前工作区。`pull` 则会先抓取，再按参数和配置整合历史。

在 `main` 上、工作区干净、只想接收可以快进的远端提交时，可以明确使用：

```bash
git pull --ff-only origin main
```

如果本地和远端已经分叉，`--ff-only` 会停止，应先理解两边的改动再决定如何整合。如果远端初始化了另一份 README，甚至可能是两条独立初始历史。不要将 `--force` 或 `--allow-unrelated-histories` 当成通用修复按钮；先核对仓库地址和历史来源。


## 8. 创建分支：让本次改动有独立的开发位置

讲义在检查点策略中给出：

```bash
git checkout -b experiment/claude-refactor
```

附录中的新功能和 Bug 修复示例分别使用 `feature/new-login`、`fix/login-timeout`。这些名称表达改动目的，属于讲义的通用示例，不能据此推断记账应用中实现了登录功能。

分支让新提交沿着独立的历史继续前进。在功能分支上开发，可以保留主分支的稳定版本，完成审查和验证后再合入主线。

**创建分支不会自动存档未提交文件。**新分支刚建立时仍指向原来的提交；工作区和暂存区中的改动也可能随切换保留下来。应先确认基线和未提交状态，再决定提交或暂存，不能把“创建了分支”当成“刚才的全部修改已保存”。

普通分支切换仍使用同一个工作目录，不会自动为每个分支生成一套独立目录。本篇使用单工作目录的分支流程。


## 9. 补充：从已同步的主线开始本次迭代

继续使用 `main` 作为实际主分支示例。先处理当前未提交修改，确认工作区干净，再按顺序操作：

```bash
git status
git switch main
git pull --ff-only origin main
git switch -c feature/ledger-improvement
git branch --show-current
```

如果拉取失败，应先解决原因，不继续把后面的命令当成成功流程执行。最后一条应显示新分支名称。

`feature/ledger-improvement` 是本篇用于串联流程的补充分支名，不是课堂实际分支名。`git switch -c` 与本场景中的 `git checkout -b` 都可创建并切换分支；旧版 Git 不支持 `switch` 时可使用讲义写法。

随后在 Claude Code 中说明本次已经确认的需求。可使用以下补充提示词，方括号部分替换为真实信息：

```text
当前项目是黑马记账，已切换到 feature/ledger-improvement。
本次只处理：[已确认的一个需求或 Bug]。
验收条件：[可观察的操作与预期结果]。
先读取 CLAUDE.md、相关实现和 package.json，说明修改范围，再实施。
沿用 Tauri + React + TypeScript 与 IndexedDB + Dexie。
只修改完成本次目标需要的文件，完成后列出改动和实际验证结果。
```

分支名负责表达版本管理范围，提示词负责表达需求；两者不能替代可观察的验收条件。


## 10. 讲义的典型开发流程

附录“新功能开发流程”的原始示例为：

```text
1. git checkout -b feature/new-login
2. git commit -m "checkpoint: before new login feature"  # 创建检查点
3. 在 Claude Code 中描述需求
4. Claude 进入 Plan Mode 设计方案
5. 确认方案后 Claude 开始编码
6. /code-review 审查改动
7. /verify 手动验证功能
8. git commit -m "feat: new login flow"
```

阅读时补齐两个前提：`git commit` 保存的是暂存区内容，因此有改动要先检查并 `git add`；若分支起点本来就是一个已提交的干净版本，不必为了名义上的检查点再制造一次空提交。

讲义的 Bug 修复流程是：创建 `fix/login-timeout` → 描述现象和复现步骤 → 定位修复 → `/code-review --fix` → `npm test` → 提交。审查流程是：完成改动 → 审查 → 根据建议修改 → 实际验证 → 提交 PR → 可选再次审查 PR。

**与记账项目的差异：**前篇已核对的 `package.json` 包含 `dev`、`build`、`preview`、`tauri`，没有确认 `test` 或 `lint` 脚本存在。这里保留讲义的测试步骤含义，但实践时应先检查脚本，不能直接宣称 `npm test` 已通过。

Plan Mode 用于先形成方案；代码审查用于检查实现；运行验证用于观察行为。三者各有作用，不能只凭“AI 说完成了”就跳到提交。


## 11. 补充：先明确要审查哪一份 diff

diff 是两个状态之间的代码差异。只说“审查一下代码”，可能遗漏已经提交的改动，也可能把整条分支的历史误当成最后一次修改。

在功能分支上，可分别检查：

| 命令 | 主要查看内容 |
|---|---|
| `git status --short` | 已修改、已暂存及未跟踪文件的清单 |
| `git diff` | 已跟踪文件中，工作区相对暂存区的差异 |
| `git diff --cached` | 暂存区相对 `HEAD` 的差异 |
| `git diff main...HEAD` | 从与本地 `main` 的共同祖先到当前提交的差异 |
| `git diff origin/main...HEAD` | 从与最近抓取的远端主分支的共同祖先到当前提交的差异 |

使用 `origin/main` 前可先 `git fetch origin`。三点比较适合查看分支引入的**已提交**改动，但不包含之后尚未提交的工作区和暂存区修改。

`git diff` 也不会自动展示未跟踪新文件的内容；应先通过 `git status` 找到它们，再读取文件。整条功能分支的审查通常需要同时覆盖：分支提交差异、暂存差异、未暂存差异和新增文件。

同理，功能分支已经推送后，如果审查只比较“本地分支相对它自己的远端上游”，可能已经没有差异。此时应明确以主分支为基线，不能把空 diff 当成整个功能已通过审查。


## 12. 代码审查检查哪些问题

讲义从五个维度说明审查目标：

| 维度 | 讲义关注点 | 在本项目中如何理解（补充） |
|---|---|---|
| 正确性 | 逻辑错误、边界遗漏、空值处理 | 金额计算、空账单、日期边界、增删改后的统计是否一致 |
| 安全性 | 注入、敏感信息、权限问题 | 用户输入怎样进入界面，是否新增非预期外部传输或泄露配置 |
| 代码复用 | 重复代码、可简化逻辑 | 新功能是否重复实现已有金额、日期和数据库处理 |
| 可读性 | 命名、结构、组织方式 | 组件职责和状态变化是否清楚，是否符合现有约定 |
| 性能 | 多余计算、泄漏、异步处理 | 筛选与图表是否重复计算，异步写入和监听是否正确处理 |

审查时应结合调用方和数据流，而不只看被修改的几行。例如修改记账写入后，要理解对应统计何时刷新，失败时界面如何响应。

讲义建议在提交前审查、大改动按文件或逻辑拆分，并由人判断是否采纳建议。对“可以更优雅”的建议和“会算错金额”的错误应区分处理，优先解决能说明触发条件和影响的实际问题。


## 13. 审查命令：讲义写法与当前适用范围

讲义列出四个入口。斜杠命令输入在 **Claude Code 会话**中，不能直接当成 PowerShell 命令运行。

| 讲义命令 | 讲义描述 | 官方文档核对后的说明 |
|---|---|---|
| `/code-review` | 审查当前工作区 diff | 当前文档提供本地差异审查，也能显式指定路径、分支或引用范围；默认还涉及相对上游未推送的提交 |
| `/review` | 审查指定 PR | 当前文档将它列为 `/code-review` 的别名；v2.1.223 以前曾是独立的 GitHub PR 审查命令 |
| `/verify` | 实际运行并验证改动 | 当前文档列为随工具提供的 skill；需要能构建、启动和操作项目的实际环境 |
| `/security-review` | 审查当前分支安全问题 | 当前文档说明比较当前分支与 `origin` 默认分支；需要能解析对应远端和基线 |

官方资料：[Commands](https://code.claude.com/docs/en/commands)、[Review a diff locally](https://code.claude.com/docs/en/code-review#review-a-diff-locally)。

工具版本、客户端及扩展会影响可用命令，先通过 `/help`、`/` 命令菜单查看本机情况。课程中的名称不能保证在任意客户端或旧版本上行为相同。

**Gitee 边界：**读取本地 Git 差异并审查，可以用于托管在 Gitee 的代码；自动查询 PR、检出 PR 和发布评论则依赖平台集成。当前官方 `--comment` 文档列出 GitHub PR、GitLab MR，没有据此确认 Gitee PR 支持。不能把 Gitee 链接直接代入讲义 `/review <PR_URL>` 后，就假定自动审查和评论一定可用。


## 14. 使用 /code-review，并核对是否覆盖本次改动

讲义保留的三个调用示例为：

```text
/code-review            # 默认审查（中等深度）
/code-review --fix      # 审查并自动修复发现的问题
/code-review --comment  # 审查并将发现发布为 PR 内联评论
```

上面的 `#` 后文字是讲义注释，实际输入只输入所需命令。

**默认深度差异：**当前官方文档说明，无显式深度时会复用之前选择的 `low` 至 `max` 级别；从未选过时使用会话的当前 effort。因此讲义的“默认中等深度”不能视为所有版本的固定行为。

当前默认范围包括当前分支领先其上游的提交，以及未提交修改。若要明确查看功能分支相对主线的已提交差异，可以在支持该语法的版本中使用以下补充示例：

```text
/code-review medium main...HEAD
```

这里的 `main` 仍需替换为实际基线。显式引用范围审查后，还要另外确认未提交修改和未跟踪文件是否被覆盖。

`--fix` 会修改工作区，修复后必须再看 diff 和验证；它不是只读报告。当前官方文档还说明，后台审查产生的修复可能不在当前会话 checkpoint 的恢复范围内，不应只依赖 `/rewind` 撤销，Git 基线仍然重要。

`--comment` 会向外部 PR/MR 发布审查内容，需要可用的平台集成与发布意图，本篇只解释，不执行。

如果命令不存在，或需要更清楚地指定范围，可用补充提示词：

```text
请审查当前功能分支相对 origin/main 的改动。
同时检查已提交差异、暂存差异、未暂存差异和未跟踪的新文件。
先说明你实际读取的范围；必要时读取调用方和相关数据处理代码。
重点找会影响正确性、安全和现有功能的问题。
每项写出文件位置、触发条件、影响和建议，并区分确定问题与待验证假设。
先给报告，不自动修改或发布评论。
```

这段提示词可用于表达本地审查目标，但仍应检查工具实际读取了什么。


## 15. 审查深度与报告：怎样判断发现是否成立

讲义用 effort 说明审查投入：

| 级别 | 讲义中的适用场景 |
|---|---|
| `low` | 小改动、快速扫描 |
| `medium` | 常规改动或 PR |
| `high` | 核心逻辑、安全敏感代码 |
| `xhigh/max` | 关键基础设施或需要更全面分析的场景 |

当前文档也支持给 `/code-review` 显式指定深度，但可用级别与模型、版本有关；更高投入并不保证没有误报或漏报。学习时可以从常规审查开始，再对金额、持久化等关键逻辑增加验证深度。本篇不扩展云端审查流程。

讲义给出的报告结构是：

```markdown
## Code Review Summary

### Critical Issues
- [file:line] 未处理的 Promise rejection 可能导致未捕获异常

### Suggestions
- [file:line] 考虑将重复的 fetch 逻辑提取为自定义 Hook
- [file:line] 建议使用 useMemo 缓存计算结果

### Praise
- 错误边界处理得很好
- 测试覆盖率充分
```

这是**讲义中的通用报告示例**，不是本项目已经获得的审查结果。不能由此认定记账项目存在上述错误，也不能认定已有充分测试。

补充判断方法：先找到实际文件和行号，再确认哪个输入或操作能触发问题，以及它是否由本次改动引入。`useMemo` 等建议应有重复计算或性能依据，不能为了迎合报告机械添加。

采纳一项修复后，重新查看修改范围，并用对应触发条件复验。报告没有发现问题，只说明在本次范围和方法下未发现问题。


## 16. 安全审查要结合应用实际能力

讲义列出的安全关注点包括 SQL 注入、XSS、CSRF、误提交 API Key 或密码、依赖漏洞、认证授权逻辑。

这些是通用检查方向，应根据应用结构选择重点。前篇的黑马记账使用本地 IndexedDB + Dexie，没有确认存在后端 SQL 数据库或登录会话，不能照搬“已检查 SQL 注入和登录权限”的结论。

对当前项目，更直接的检查是：输入和导入内容是否被当作不可信数据处理；是否把内容送入危险的 HTML 渲染入口；源码及历史中是否出现凭据；是否意外引入外部请求；若本次改动涉及 Tauri 原生能力，权限范围是否符合需要。

在提供该命令的 Claude Code 环境中：

```text
/security-review
```

当前官方说明依赖 `origin` 及其默认分支。出现基线解析错误时，应先检查远端和分支状态，不能把失败当成“没有安全问题”。该命令的结果也不能替代单独的依赖漏洞扫描结果或实际行为验证。


## 17. /verify：运行和观察，确认功能确实生效

讲义强调，代码审查之后还要实际验证：前端改动启动应用查看界面，API 改动发送请求，CLI 改动运行命令观察结果。

```text
/verify
```

在支持该命令且具备相应运行能力的环境中，它用于构建、启动并观察应用。不能把命令已输入或构建通过，直接当成所有功能已验证。

**补充：记账项目的验证分层。**在应用项目根目录，先检查 `package.json` 的真实脚本。前篇已核对的构建命令是：

```bash
npm run build
```

它执行 TypeScript 检查和 Vite 前端构建。验证界面交互时，可按前篇方式启动开发环境：

```bash
npm run dev
```

若本次改动涉及 Tauri 原生窗口、文件能力或桌面运行差异，应在桌面开发环境中继续观察，而不是只看浏览器页面；前篇使用的入口为：

```bash
npx tauri dev
```

需要交付桌面包时再执行对应打包验证：

```bash
npx tauri build
```

以上是**待执行的课程实践命令**，本次编写笔记没有运行它们。是否需要打包，应由本次改动和交付目标决定。

围绕实际需求检查操作结果，并保留已有核心功能的回归观察。记账项目可关注新增、编辑、删除后记录与统计的一致性，空数据状态，以及重启后的持久化；改到筛选、图表或 Excel 功能时，再针对这些行为验证。使用测试数据，不能把真实账目当作可随意覆盖的数据。

记录时写清“执行了什么、观察到什么、还有什么没覆盖”。如果没有 `test` 脚本、没有桌面运行环境或无法观察某个交互，应直接写出限制。


## 18. 补充：审查修复后提交，再推送功能分支

前提是仍在 `feature/ledger-improvement`，审查问题已经处理，并完成相应验证。先检查本次变更：

```bash
git branch --show-current
git status --short
git diff
git diff --cached
```

随后只暂存本次任务涉及的文件。若确认当前项目中所有待变更文件都属于本次任务，才使用：

```bash
git add -A
git diff --cached --stat
git diff --cached
git commit -m "feat: improve ledger workflow"
```

提交说明也是补充示例，应替换为真实修改目的；修复 Bug 时可使用相应的 `fix:` 描述。若暂存 diff 与刚才验证的代码不一致，应先查清原因再提交。

首次推送这个功能分支：

```bash
git push -u origin feature/ledger-improvement
git status -sb
git log -1 --oneline
```

打开 Gitee，切到对应功能分支，核对最新提交。此时推送的是功能分支，主分支不会因为它已上传就自动包含该功能。

讲义建议“提交前审查”。即使先提交了一个本地检查点，也要在进入主线前覆盖整条功能分支的改动，不能只审查最后一次修复产生的几行差异。


## 19. 补充：用 Gitee PR 承接审查与合并

PR（Pull Request）是请求将一个分支的修改合入另一个分支的协作入口。它承载差异、讨论与验证信息，和终端中的 `git pull` 命令不是同一件事。

Gitee 官方 [合并分支](https://help.gitee.com/repository/branch/merge) 说明了如下流程：

1. 在仓库中切到待合并的分支，通过“贡献代码”等入口创建 Pull Request。
2. 确认源分支与目标分支。本篇补充示例的源为 `feature/ledger-improvement`，目标为实际主分支 `main`。
3. 填写说明，检查提交和文件差异，完成审查与测试后再合并。

源分支是提供修改的一方，目标分支是接收修改的一方，不要反选。PR 说明应包含：解决了什么问题、改动后的行为、实际验证方式，以及尚未验证的部分。

讲义使用 `/review <PR_URL>` 表示再次审查 PR 的意图。对 Gitee，应先确认当前工具有相应集成；没有时，可审查本地以主线为基准的差异，再由人在 Gitee 页面查看和处理 PR，不假定 GitHub 的自动评论能力可直接迁移。

同仓库的功能分支 PR 不要求先 fork。Gitee 的 [Pull Request 简介](https://help.gitee.com/base/pullrequest/intro) 还介绍了给他人仓库贡献时的 fork 流程，那是另一种协作场景。

PR 确认已在 Gitee 合并后，工作区干净时更新本地主线：

```bash
git switch main
git pull --ff-only origin main
git log -5 --oneline
```

如果快进更新失败，应先检查本地主线是否还有未整合的独立提交。合并产生了新的组合状态时，也应针对受影响的功能再确认行为，而不只依赖合并前的审查结论。


## 20. 本篇来源与实践状态

课程来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：核对视频 14、15 的完整名称及相邻课程顺序，检查资料目录；未取得视频逐段转录。
- [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p)：核对分支示例、代码审查维度、原始命令、报告示例及附录流程。

用于补充操作与核对差异的官方资料：

- [Pro Git：远程仓库的使用](https://git-scm.com/book/zh/v2/Git-基础-远程仓库的使用)：远端简称、地址、fetch、pull、push。
- [Pro Git：分支的新建与合并](https://git-scm.com/book/zh/v2/Git-分支-分支的新建与合并)：创建分支、切换前的工作区状态、合并。
- [Git push 文档](https://git-scm.com/docs/git-push)：显式推送分支与上游关联参数。
- [Gitee：账户 SSH 公钥设置](https://help.gitee.com/base/account/SSH%E5%85%AC%E9%92%A5%E8%AE%BE%E7%BD%AE)：账号认证、账户公钥与部署公钥的权限区别。
- [Gitee：合并分支](https://help.gitee.com/repository/branch/merge)：源分支、目标分支和 PR 合并流程。
- [Claude Code：Commands](https://code.claude.com/docs/en/commands)、[本地 diff 审查](https://code.claude.com/docs/en/code-review#review-a-diff-locally)：审查范围、别名、深度、自动修复与评论发布能力。

本次只新增本篇 Notebook，没有为记账应用新建 Gitee 仓库、配置凭据、修改 Git 远端、提交、推送、创建分支、创建或合并 PR，也没有执行 Claude Code 命令和应用构建。正文命令用于学习，不能当作已执行结果。

相关讲义以文字、表格和代码为主，没有需要保留的原始教学插图，本篇不添加装饰图。
